## Análisis exploratorio de la caida en la Competencia Lectora de los informes PISA en los últimos años  (2018 -> 2022 -> 2025)

## 1. Introducción

PISA es la evaluación internacional de la OCDE que mide cada tres años las competencias
del alumnado de 15 años. En la Comunidad Valenciana, la puntuación media en lectura pasó
de 481,8 en 2022 a 423,5 en 2025. Este proyecto explora **si esa caída puede explicarse
por cambios en el perfil del alumnado o en el uso de tecnología en el centro**.

**Preguntas:** 
1. Dentro de la Comunidad Valenciana, para el año de 2025, ¿que variables están más correlacionadas con una menor nota en lectura?


2. ¿Qué factores observables en los microdatos de PISA acompañan a la caída
de la competencia lectora  de 2018 a 2025 en la Comunidad Valenciana?


**Hipótesis de partida**:
- H1.1. El nivel socioeconómico (ESCS) del alumnado condiciona la nota en la competencia lectora.
- H1.2. La titularidad del centro (público/privado) del alumnado condiciona la nota en la competencia lectora.
- H2. Han cambiado la composición por origen migratorio y del idioma en que se realiza la prueba.

**Alcance y límites:** análisis exploratorio, descriptivo y sin conclusiones causales,
con una muestra de unos 50 centros por ciclo.

**Hipótesis interesantes que no da tiempo a estudiar correctamente**
- H1.3. El tiempo de pantalla invertido en los centros ha aumentado, asociandose con la caida en la competencia lectora.
- H1.4. El tiempo de pantalla invertido por los alumnos fuera del aula ha aumentado, asociandose con la caida en la competencia lectora.
- H1.5. Las metodologías docentes en los centros ha cambiado, influyendo en la caida en la competencia lectora.

## 2. Fuentes de datos

Microdatos del cuestionario de alumnado de PISA 2018, 2022 y 2025 (OCDE, ficheros SPSS):
`CY07_MSU_STU_QQQ.sav`, `CY08MSP_STU_QQQ.SAV` y `CY09_MS_STU_PUF.sav`, y los codebooks correcpondientes: [https://www.oecd.org/en/data/datasets/pisa-2018-database.html#codebook]

- **Unidad de análisis:** alumno de 15 años. 
- **Diseño muestral:** primero centros y luego alumnos dentro de cada centro, por lo que hay que usar pesos (`W_FSTUWT`) y pesos replicados (`W_FSTURWT1-80`) para los errores.
- **Competencia lectora:** 10 valores plausibles (`PV1READ`–`PV10READ`).
- **Muestra de la Comunidad Valenciana:** 1.753 / 1.625 / 1.712 alumnos en 53 / 53 / 54
  centros (2018 / 2022 / 2025).
- **Variables de contexto:** ESCS, origen migratorio (`IMMIG`), idioma hablado en casa (`LANGN`),
  idioma en que se realiza la prueba (`LANGTEST_COG`), horas de uso digital en el centro (`ST326Q01JA`).

In [2]:
#Librerias
import pyreadstat
import pandas as pd
import numpy as np

In [1]:
# Definimos las variables que estudiaremos en el análisis
cols = (["CNT","STRATUM", "CNTSCHID", "CNTSTUID", "ST326Q01JA",
            "ST326Q05JA", "ESCS", "W_FSTUWT", "LANGN", "LANGTEST_COG", "IMMIG"]
           + [f"PV{i}READ" for i in range(1, 11)] + [f"ST438Q0{i}DA" for i in range(1,5)]+ [f"W_FSTURWT{i}" for i in range(1, 81)])

cols_pantallas_18 = [f"ST176Q0{i}IA" for i in range(1, 8) ]

cols18 = cols+cols_pantallas_18 

In [3]:
# Cargamos los datos filtrados por las columnas que queremos y por la comunidad valenciana

archivo_pisa25 = 'data/CY09_MS_STU_PUF.sav'
print("Cargando los archivo de microdatos de PISA 2025 (puede tardar unos minutos)...")
codigos_valencia25 = ['ESP33', 'ESP34']

archivo_pisa22="data/CY08MSP_STU_QQQ.SAV"
print("Cargando los archivo de microdatos de PISA 2022 (puede tardar unos minutos)...")
codigos_valencia22 = ['ESP34', 'ESP35']

archivo_pisa18="data/CY07_MSU_STU_QQQ.sav"
print("Cargando los archivo de microdatos de PISA 2018 (puede tardar unos minutos)...")
codigos_valencia18 = ['ESP9038', 'ESP1738', 'ESP9039', 'ESP1739']

def cargar_valencia(ruta, estratos, cols):
    # 1) Leemos solo las columnas que necesitamos
    df, meta = pyreadstat.read_sav(ruta, usecols=cols)

    # 2) Nos quedamos con los alumnos de España y de los estratos de Valencia
    d = df[(df["CNT"] == "ESP") & (df["STRATUM"].isin(estratos))]
    
    # 3) Algunas comprobaciones en la práctica: tamaño y nº de pesos replicados
    n_rep = len([c for c in d.columns if "RWT" in c])
    print("Filas:", d.shape[0], "| centros:", d["CNTSCHID"].nunique())
    return d, meta

df_valencia25, meta25 = cargar_valencia(archivo_pisa25, codigos_valencia25, cols)
df_valencia22, meta22 = cargar_valencia(archivo_pisa22, codigos_valencia22, cols)
df_valencia18, meta18 = cargar_valencia(archivo_pisa18, codigos_valencia18, cols18)



Cargando los archivo de microdatos de PISA 2025 (puede tardar unos minutos)...
Cargando los archivo de microdatos de PISA 2022 (puede tardar unos minutos)...
Cargando los archivo de microdatos de PISA 2018 (puede tardar unos minutos)...
Filas: 1712 | centros: 54
Filas: 1625 | centros: 53
Filas: 1753 | centros: 53


In [4]:
# Pasamos a csv los df cargados y creados 
df_valencia25.to_csv("data/pisa_valencia25.csv")
df_valencia22.to_csv("data/pisa_valencia22.csv")
df_valencia18.to_csv("data/pisa_valencia18.csv")


In [ ]:
pd.set_option("display.max_columns", None) 
df_valencia22.head(10) 

,CNT,CNTSCHID,CNTSTUID,STRATUM,LANGTEST_COG,ST326Q01JA,ST326Q05JA,IMMIG,LANGN,ESCS,W_FSTUWT,W_FSTURWT1,W_FSTURWT2,W_FSTURWT3,W_FSTURWT4,W_FSTURWT5,W_FSTURWT6,W_FSTURWT7,W_FSTURWT8,W_FSTURWT9,W_FSTURWT10,W_FSTURWT11,W_FSTURWT12,W_FSTURWT13,W_FSTURWT14,W_FSTURWT15,W_FSTURWT16,W_FSTURWT17,W_FSTURWT18,W_FSTURWT19,W_FSTURWT20,W_FSTURWT21,W_FSTURWT22,W_FSTURWT23,W_FSTURWT24,W_FSTURWT25,W_FSTURWT26,W_FSTURWT27,W_FSTURWT28,W_FSTURWT29,W_FSTURWT30,W_FSTURWT31,W_FSTURWT32,W_FSTURWT33,W_FSTURWT34,W_FSTURWT35,W_FSTURWT36,W_FSTURWT37,W_FSTURWT38,W_FSTURWT39,W_FSTURWT40,W_FSTURWT41,W_FSTURWT42,W_FSTURWT43,W_FSTURWT44,W_FSTURWT45,W_FSTURWT46,W_FSTURWT47,W_FSTURWT48,W_FSTURWT49,W_FSTURWT50,W_FSTURWT51,W_FSTURWT52,W_FSTURWT53,W_FSTURWT54,W_FSTURWT55,W_FSTURWT56,W_FSTURWT57,W_FSTURWT58,W_FSTURWT59,W_FSTURWT60,W_FSTURWT61,W_FSTURWT62,W_FSTURWT63,W_FSTURWT64,W_FSTURWT65,W_FSTURWT66,W_FSTURWT67,W_FSTURWT68,W_FSTURWT69,W_FSTURWT70,W_FSTURWT71,W_FSTURWT72,W_FSTURWT73,W_FSTURWT74,W_FSTURWT75,W_FSTURWT76,W_FSTURWT77,W_FSTURWT78,W_FSTURWT79,W_FSTURWT80,PV1READ,PV2READ,PV3READ,PV4READ,PV5READ,PV6READ,PV7READ,PV8READ,PV9READ,PV10READ
487800,ESP,72400423.0,72400033.0,ESP34,156.0,2.0,3.0,1.0,608.0,1.1647,31.03820,15.51910,15.51910,15.51910,15.51910,15.51910,46.55731,46.55731,15.51910,46.55731,46.55731,15.51910,15.51910,46.55731,46.55731,46.55731,46.55731,15.51910,46.55731,15.51910,46.55731,46.55731,46.55731,46.55731,46.55731,46.55731,15.51910,15.51910,46.55731,15.51910,15.51910,46.55731,46.55731,15.51910,15.51910,15.51910,15.51910,46.55731,15.51910,46.55731,15.51910,15.51910,15.51910,15.51910,15.51910,15.51910,46.55731,46.55731,15.51910,46.55731,46.55731,15.51910,15.51910,46.55731,46.55731,46.55731,46.55731,15.51910,46.55731,15.51910,46.55731,46.55731,46.55731,46.55731,46.55731,46.55731,15.51910,15.51910,46.55731,15.51910,15.51910,46.55731,46.55731,15.51910,15.51910,15.51910,15.51910,46.55731,15.51910,46.55731,15.51910,430.478,476.122,448.517,437.518,495.827,496.781,486.951,441.814,447.966,477.489
487806,ESP,72400706.0,72400039.0,ESP35,156.0,4.0,5.0,1.0,156.0,0.8304,34.37007,15.84579,16.44818,16.94766,52.09819,54.42577,56.79249,50.32921,18.27546,46.44020,18.27546,48.46656,17.00891,15.84579,15.35044,16.94766,54.57948,50.32921,17.66778,54.57948,52.58262,54.42577,54.57948,52.58262,17.00891,15.84579,15.35044,17.56255,46.44020,18.27546,46.44020,17.66778,52.09819,54.42577,56.79249,52.58262,16.44818,17.56255,48.46656,16.44818,16.94766,15.84579,16.44818,16.94766,52.09819,54.42577,56.79249,50.32921,18.27546,46.44020,18.27546,48.46656,17.00891,15.84579,15.35044,16.94766,54.57948,50.32921,17.66778,54.57948,52.58262,54.42577,54.57948,52.58262,17.00891,15.84579,15.35044,17.56255,46.44020,18.27546,46.44020,17.66778,52.09819,54.42577,56.79249,52.58262,16.44818,17.56255,48.46656,16.44818,16.94766,522.658,556.093,355.133,455.137,431.385,413.876,550.085,456.583,494.628,399.648
487813,ESP,72400463.0,72400047.0,ESP35,156.0,6.0,4.0,1.0,156.0,0.8970,36.40645,54.60967,18.20322,18.20322,18.20322,14.90011,66.35820,14.90011,66.35820,14.90011,54.60967,54.60967,54.60967,66.35820,18.20322,14.90011,66.35820,18.20322,14.90011,54.60967,66.35820,18.20322,54.60967,54.60967,54.60967,66.35820,14.90011,66.35820,14.90011,66.35820,18.20322,18.20322,18.20322,14.90011,54.60967,66.35820,14.90011,54.60967,66.35820,18.20322,14.90011,54.60967,18.20322,18.20322,18.20322,14.90011,66.35820,14.90011,66.35820,14.90011,54.60967,54.60967,54.60967,66.35820,18.20322,14.90011,66.35820,18.20322,14.90011,54.60967,66.35820,18.20322,54.60967,54.60967,54.60967,66.35820,14.90011,66.35820,14.90011,66.35820,18.20322,18.20322,18.20322,14.90011,54.60967,66.35820,14.90011,54.60967,66.35820,18.20322,14.90011,599.042,586.149,639.843,643.675,643.622,669.051,616.782,624.996,604.314,625.888
487824,ESP,72400402.0,72400059.0,ESP34,156.0,2.0,2.0,1.0,852.0,0.8246,27.49415,42.72057,42.72057,42.72057,13.28763,13.28763,13.28763,13.28763,42.72057,13.28763,42.72057,13.28763,42.72057,42.72057,42.72057,42.72057,13.28763,13.28763,42.7205

In [48]:
print("Filas y columnas 2025:", df_valencia25.shape)
print("Columnas con nulos 2025:", df_valencia25.isna().sum()[df_valencia25.isna().sum() > 0].to_dict())
print("Filas duplicadas 2025:", df_valencia25.duplicated().sum())
print("Alumnos id repetidos 2025:", df_valencia25["CNTSTUID"].duplicated().sum())
print("-----------------------------------------------------------------------------")
print("Filas y columnas 2022:", df_valencia22.shape)
print("Columnas con nulos 2022:", df_valencia22.isna().sum()[df_valencia22.isna().sum() > 0].to_dict())
print("Filas duplicadas 2022:", df_valencia22.duplicated().sum())
print("Alumnos id repetidos 2022:", df_valencia22["CNTSTUID"].duplicated().sum())
print("-----------------------------------------------------------------------------")
print("Filas y columnas 2018:", df_valencia18.shape)
print("Columnas con nulos 2018:", df_valencia18.isna().sum()[df_valencia18.isna().sum() > 0].to_dict())
print("Filas duplicadas 2018:", df_valencia18.duplicated().sum())
print("Alumnos id repetidos 2018:", df_valencia18["CNTSTUID"].duplicated().sum())

Filas y columnas 2025: (1712, 105)
Columnas con nulos 2025: {'ST326Q01JA': 239, 'ST326Q05JA': 255, 'ST438Q01DA': 220, 'ST438Q02DA': 225, 'ST438Q03DA': 231, 'ST438Q04DA': 228, 'ESCS': 92, 'IMMIG': 95, 'LANGN': 61}
Filas duplicadas 2025: 0
Alumnos id repetidos 2025: 0
-----------------------------------------------------------------------------
Filas y columnas 2022: (1625, 101)
Columnas con nulos 2022: {'ST326Q01JA': 46, 'ST326Q05JA': 64, 'IMMIG': 63, 'ESCS': 31}
Filas duplicadas 2022: 0
Alumnos id repetidos 2022: 0
-----------------------------------------------------------------------------
Filas y columnas 2018: (1753, 105)
Columnas con nulos 2018: {'ST176Q01IA': 110, 'ST176Q02IA': 116, 'ST176Q03IA': 117, 'ST176Q05IA': 115, 'ST176Q06IA': 122, 'ST176Q07IA': 116, 'IMMIG': 59, 'ESCS': 34}
Filas duplicadas 2018: 0
Alumnos id repetidos 2018: 0


A. Verificamos que los estratos escogidos son los de Valencia :

In [6]:
for anio, m, cods in [("2018", meta18, codigos_valencia18),
                      ("2022", meta22, codigos_valencia22),
                      ("2025", meta25, codigos_valencia25)]:
    etiquetas = m.variable_value_labels.get("STRATUM", {})
    for c in cods:
        print(anio, c, "->", etiquetas.get(c, "SIN ETIQUETA"))

2018 ESP9038 -> ESP - stratum 38: Valencia, public
2018 ESP1738 -> ESP - stratum 38: Valencia, public
2018 ESP9039 -> ESP - stratum 39: Valencia, private
2018 ESP1739 -> ESP - stratum 39: Valencia, private
2022 ESP34 -> ESP - stratum 34: Comunidad Valenciana,Public
2022 ESP35 -> ESP - stratum 35: Comunidad Valenciana,Private
2025 ESP33 -> Comunidad Valenciana, Public
2025 ESP34 -> Comunidad Valenciana, Private


## Resumen de las decisiones de limpieza, que se llevará a cabo en cada uno de los análisis de cada variable:
- No se imputan valores perdidos: se excluyen solo en el cálculo que usa cada variable.
- No se eliminan centros pequeños ni alumnos atípicos: los pesos ya recogen el diseño.
- No se eliminan duplicados porque no hay ninguno (CNTSTUID único).